# Loading a Filled Dataset from Datacube  
The `load_filled_grid` function in the `utils.extract_filled_grid` module loads data from the datacube for the specified bounding box and time range provided as input parameters. The function returns an xarray Dataset containing the filled grid data for the specified area and time period, with a time slice for the entite temporal range.

Knowing that Earh observation data from optical sensors is prone to gaps due to cloud cover, the `load_filled_grid` function applies temporal interpolation to fill these gaps, providing a complete time series of data suitable for analysis. This same function also handles gaps where is no data at all for a single time slice as there was no overpass on that day. 

The gap-filling method used here is **DINEOF** (Data Interpolating Empirical Orthogonal Functions), and it fills gaps the same way you'd reconstruct a blurry photo from its most important features.

### Setup  
1. The spatial grid is flattened to a 2D matrix: rows = time steps, columns = ocean pixels
2. A small fraction of known values are secretly withheld (cross-validation set) so we can measure how well the reconstruction is doing without peeking at the actual gaps

### Finding the right number of patterns (auto mode)  
3. The algorithm tests different numbers of "modes" (1, 3, 5... up to 20) by running a quick version of steps below, measuring error against the withheld values each time, and picking whichever gives the lowest error

### Iterative reconstruction  
4. Gaps are given an initial guess: the long-term average value at that location
5. Then repeatedly:
    - Run PCA (Principal Component Analysis) on the filled matrix — this finds the dominant spatial patterns and how they vary over time (e.g. "there's always a warm patch in the northwest in summer")
    - Reconstruct the whole matrix using only the top N patterns — this smooths out noise and produces plausible values everywhere
    - Replace only the gap pixels with the reconstructed values; keep real observations as-is
    - Check whether the reconstructed values at the withheld pixels are getting better or worse
    - Stop when improvement drops below the tolerance threshold

#### Why it works  
Ocean variables are strongly correlated in space and time — temperature in one pixel is similar to its neighbours, and follows seasonal patterns. PCA captures those correlations. By reconstructing from a limited number of patterns, the algorithm effectively borrows information from nearby pixels and other time steps to make an educated guess at the missing value.

### After DINEOF  
6. The withheld cross-validation values are restored to their true values
7. Any ocean pixels that are still NaN (e.g. a location with zero observations across the entire time series) are filled with the temporal mean as a last resort
8. Land pixels are left as NaN throughout

#### Limitations
The interpoloation method requires at least some data to run; if there are no data available, perhaps because the AoI is too small or the temporal range is relatively short, the algorithm will not be able to fill the gaps effectively and may return suboptimal results or fail entirely.


## Load data
Start by defining the Area of Interest (AoI) as a bounding box as a list, using the format `[min_lon, min_lat, max_lon, max_lat]` or `[west, south, east, north]`:

In [1]:
bbox=[10.70,56.86,12.68,58.78]

then call the function providing a date range and, optionally, a resolution. Datacube handles the reprojection natively ensuring that all the variables in the resulting datase are on a common grid. Specifying `verbose=True` provides feedback about progress and timings:

In [2]:
from utils.extract_filled_grid import load_filled_grid

combined_data = load_filled_grid(
    bbox=bbox,
    time_range=('2020-04-01', '2020-04-30'),
    # resolution=0.125,
    verbose=True
)
combined_data

Loading SSS (cmems_sss)...
  SSS loaded in 1.7s
Loading SST (s3_slstr_sst)...
  SST loaded in 1.1s
Loading CHL (cmems_chl_tur)...
  CHL loaded in 0.0s
Aligned to 30 daily time steps (2020-04-01 – 2020-04-30)
Ocean mask: 186/289 pixels (64.4% ocean)
Computing data arrays...
  Compute done in 24.8s
Starting DINEOF gap-filling...
  Processing salinity...
    Data shape: 30 time steps x 186 ocean points
    Initial coverage: 4740/5580 (84.95%)
    Cross-validation: 237 points withheld (5.0%)
    Determining optimal number of EOF modes...
    Optimal number of modes: 7
    Starting DINEOF iterations (max: 100)...
      Iteration 10: CV-RMSE = 0.231110, Change = 0.004142
    Converged at iteration 15 (CV-RMSE = 0.226703)
    Final coverage (ocean only): 100.00%
  Salinity done in 0.7s
  Processing temperature...
    Data shape: 30 time steps x 186 ocean points
    Initial coverage: 3037/5580 (54.43%)
    Cross-validation: 151 points withheld (5.0%)
    Determining optimal number of EOF modes

<xarray.Dataset> Size: 209kB
Dimensions:      (time: 30, latitude: 17, longitude: 17)
Coordinates:
  * time         (time) datetime64[ns] 240B 2020-04-01 2020-04-02 ... 2020-04-30
  * latitude     (latitude) float64 136B 58.81 58.69 58.56 ... 57.06 56.94 56.81
  * longitude    (longitude) float64 136B 10.69 10.81 10.94 ... 12.56 12.69
    spatial_ref  int32 4B 4326
Data variables:
    salinity     (time, latitude, longitude) float64 69kB 28.33 28.16 ... nan
    temperature  (time, latitude, longitude) float64 69kB 5.055 5.06 ... nan
    chlorophyll  (time, latitude, longitude) float64 69kB 6.177 5.731 ... nan
    ocean_mask   (latitude, longitude) bool 289B True True True ... True False

## Data Visualisation

It may be helpful to visualise the three variables from the combined dataset. The cell below will present the first day in the temporal range with a date picker to navigate through time.  

In [3]:
import pandas as pd
from ipywidgets import interact, DatePicker
from utils.extract_filled_grid import plot_timestep

min_date=pd.Timestamp(combined_data.time.min().values)
max_date=pd.Timestamp(combined_data.time.max().values)

date_picker = DatePicker(
    description='Select Date:',
    value=pd.Timestamp(combined_data.time.min().values),
    min=min_date,
    max=max_date,
)

print("Select a date to visualise the data for that day:")
print(f"\n=== Available Date Range ===")
print(f"Start Date:  {min_date}")
print(f"End Date:    {max_date}\n")

interact(lambda selected_date: plot_timestep(combined_data, selected_date, False), selected_date=date_picker)
print("")

Select a date to visualise the data for that day:

=== Available Date Range ===
Start Date:  2020-04-01 00:00:00
End Date:    2020-04-30 00:00:00



interactive(children=(DatePicker(value=Timestamp('2020-04-01 00:00:00'), description='Select Date:', max=Times…